# Predicting U.S. Recessions with Macroeconomic Indicators

**Portfolio econometrics / data science project**

This project asks a practical forecasting question:

> **Can publicly available macroeconomic indicators help identify whether a U.S. recession will begin or be underway within the next 12 months?**

Five increasingly rich logistic-regression specifications are compared:

| Model | Predictors |
|---|---|
| 1 | 10Y–2Y Treasury yield spread |
| 2 | + 3-month unemployment-rate change |
| 3 | + 12-month housing-start growth |
| 4 | + 12-month consumer-sentiment change |
| 5 | + Baa–10Y Treasury credit spread |

The analysis emphasizes **out-of-sample evaluation**, **time ordering**, and **leakage-aware walk-forward validation** rather than maximizing in-sample fit.

### Skills demonstrated
Python · pandas · FRED API · feature engineering · logistic regression · time-series validation · ROC-AUC · precision/recall · data visualization

> This is an educational forecasting exercise, not investment advice or an official recession-dating model.


## 1. Setup

Install the required packages if needed:

`python -m pip install pandas numpy matplotlib scikit-learn fredapi python-dotenv`

Create a `.env` file in the same project folder:

`FRED_API_KEY=your_key_here`


In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from fredapi import Fred
from dotenv import load_dotenv

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score,
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
)

warnings.filterwarnings("ignore", category=FutureWarning)

load_dotenv()
api_key = os.getenv("FRED_API_KEY")

if api_key is None:
    raise ValueError(
        "FRED_API_KEY was not found. Create a .env file containing "
        "FRED_API_KEY=your_key_here"
    )

fred = Fred(api_key=api_key)

pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:0.3f}")


## 2. Data

### FRED series

- `USREC` — NBER-based U.S. recession indicator
- `T10Y2Y` — 10-year Treasury yield minus 2-year Treasury yield
- `UNRATE` — civilian unemployment rate
- `HOUST` — housing starts
- `UMCSENT` — University of Michigan consumer sentiment
- `BAA10Y` — Moody's Baa corporate bond yield relative to the 10-year Treasury yield

The raw series have different frequencies. The daily yield spread is converted to a monthly average so every observation represents one month.


In [ ]:
# Download series
recession = fred.get_series("USREC").rename("Recession")
yield_spread = fred.get_series("T10Y2Y").rename("Yield_Spread")
unemployment = fred.get_series("UNRATE").rename("Unemployment")
housing = fred.get_series("HOUST").rename("Housing_Starts")
sentiment = fred.get_series("UMCSENT").rename("Consumer_Sentiment")
credit_spread = fred.get_series("BAA10Y").rename("Credit_Spread")

# Convert daily series to monthly averages
yield_spread = yield_spread.resample("MS").mean()
credit_spread = credit_spread.resample("MS").mean()

# Combine into one monthly DataFrame
data = pd.concat(
    [
        recession,
        yield_spread,
        unemployment,
        housing,
        sentiment,
        credit_spread,
    ],
    axis=1,
)

# Begin when the 10Y-2Y spread is available
data = data.loc["1976-06-01":].copy()

data.tail()


## 3. Feature engineering

Raw macroeconomic levels are not always the most informative predictors. We create economically interpretable changes:

- **Unemployment change (3M):** current unemployment minus unemployment three months earlier.
- **Housing growth (12M):** year-over-year percentage growth in housing starts.
- **Sentiment change (12M):** current sentiment minus sentiment one year earlier.
- **Credit spread:** Baa corporate yield minus the 10-year Treasury yield, already expressed as a spread by FRED.

Expected associations with recession risk:

- Yield spread: **negative**
- Unemployment change: **positive**
- Housing growth: **negative**
- Sentiment change: **negative**
- Credit spread: **positive**


In [ ]:
data["Unemployment_Change_3M"] = (
    data["Unemployment"] - data["Unemployment"].shift(3)
)

data["Housing_Growth_12M"] = (
    data["Housing_Starts"].pct_change(12, fill_method=None) * 100
)

data["Sentiment_Change_12M"] = (
    data["Consumer_Sentiment"] - data["Consumer_Sentiment"].shift(12)
)

data[
    [
        "Yield_Spread",
        "Unemployment_Change_3M",
        "Housing_Growth_12M",
        "Sentiment_Change_12M",
        "Credit_Spread",
    ]
].tail()


## 4. Define the forecasting target

For month \(t\), the target is:

**1** if `USREC = 1` in any of months t+1,…,t+12, otherwise **0**.

This asks whether a recession occurs **within the next 12 months**, rather than whether the economy is already in recession.


In [ ]:
future_recession = (
    data["Recession"]
    .shift(-1)
    .rolling(window=12)
    .max()
    .shift(-11)
)

data["Recession_Next_12M"] = future_recession

data[
    ["Recession", "Recession_Next_12M"]
].dropna().tail(20)


## 5. Exploratory visualization


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(data.index, data["Yield_Spread"], label="10Y - 2Y Treasury Spread")
ax.axhline(0, linestyle="--", linewidth=1)

valid = data["Recession"].notna() & data["Yield_Spread"].notna()

ax.fill_between(
    data.index,
    data["Yield_Spread"].min(skipna=True),
    data["Yield_Spread"].max(skipna=True),
    where=(data["Recession"] == 1) & valid,
    alpha=0.15,
    label="NBER Recession",
)

ax.set_title("Treasury Yield Spread and U.S. Recessions")
ax.set_xlabel("Year")
ax.set_ylabel("Percentage points")
ax.legend()
plt.show()


## 6. Five model specifications

The models are **nested**: each specification adds one new source of macroeconomic information.

Standardization is included inside each model pipeline. This is important because predictors use different units. It also makes coefficient magnitudes more comparable.

The scaler is fit **only on the training data** inside each validation fold, avoiding preprocessing leakage.


In [ ]:
MODEL_SPECS = {
    "Model 1": [
        "Yield_Spread",
    ],
    "Model 2": [
        "Yield_Spread",
        "Unemployment_Change_3M",
    ],
    "Model 3": [
        "Yield_Spread",
        "Unemployment_Change_3M",
        "Housing_Growth_12M",
    ],
    "Model 4": [
        "Yield_Spread",
        "Unemployment_Change_3M",
        "Housing_Growth_12M",
        "Sentiment_Change_12M",
    ],
    "Model 5": [
        "Yield_Spread",
        "Unemployment_Change_3M",
        "Housing_Growth_12M",
        "Sentiment_Change_12M",
        "Credit_Spread",
    ],
}

ALL_FEATURES = MODEL_SPECS["Model 5"]

model_data = data.dropna(
    subset=ALL_FEATURES + ["Recession_Next_12M"]
).copy()

print("Usable monthly observations:", len(model_data))
print("First observation:", model_data.index.min().date())
print("Last observation:", model_data.index.max().date())


## 7. Initial holdout comparison

For continuity with the exploratory analysis, models are first trained through December 2005 and evaluated from January 2006 onward.

This is **not** the final model-selection test because examining the same holdout repeatedly can overfit research decisions to that period. The more rigorous comparison comes later using purged walk-forward validation.


In [ ]:
train = model_data.loc[:"2005-12-01"].copy()
test = model_data.loc["2006-01-01":].copy()

holdout_results = []
holdout_probabilities = {}

for model_name, features in MODEL_SPECS.items():
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("logit", LogisticRegression(max_iter=1000)),
    ])

    pipeline.fit(
        train[features],
        train["Recession_Next_12M"],
    )

    probabilities = pipeline.predict_proba(test[features])[:, 1]
    auc = roc_auc_score(test["Recession_Next_12M"], probabilities)

    holdout_probabilities[model_name] = probabilities

    holdout_results.append([
        model_name,
        len(features),
        auc,
    ])

holdout_table = pd.DataFrame(
    holdout_results,
    columns=["Model", "Number_of_Predictors", "Holdout_ROC_AUC"],
)

holdout_table


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

for model_name in MODEL_SPECS:
    ax.plot(
        test.index,
        holdout_probabilities[model_name],
        label=model_name,
        linewidth=1.5,
    )

ax.fill_between(
    test.index,
    0,
    1,
    where=(test["Recession_Next_12M"] == 1),
    alpha=0.10,
    label="Recession within next 12 months",
)

ax.set_title("Out-of-Sample Recession Probabilities: Five Models")
ax.set_xlabel("Year")
ax.set_ylabel("Predicted probability")
ax.set_ylim(0, 1)
ax.legend(ncol=2)
plt.show()


## 8. Standardized coefficient comparison

Because every predictor is standardized before logistic regression, these coefficients correspond to a one-standard-deviation increase in a predictor, holding the others fixed.

A positive coefficient raises estimated recession log-odds; a negative coefficient lowers them. Coefficients describe conditional associations in this model and should **not** be interpreted as causal effects.

## 8.1 Econometric Inference for the Full Specification

The primary objective of this project is out-of-sample prediction rather than causal inference. However, as a complementary diagnostic, the full logistic specification is also estimated using `statsmodels` to examine coefficient uncertainty and statistical significance.

These p-values and confidence intervals should be interpreted cautiously because the monthly observations are time-series data and the 12-month-ahead target creates substantial overlap across adjacent observations. Statistical significance is therefore not used for model selection.


In [ ]:
coefficient_rows = []

for model_name, features in MODEL_SPECS.items():
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("logit", LogisticRegression(max_iter=1000)),
    ])

    pipeline.fit(
        train[features],
        train["Recession_Next_12M"],
    )

    coefficients = pipeline.named_steps["logit"].coef_[0]

    for feature, coefficient in zip(features, coefficients):
        coefficient_rows.append([
            model_name,
            feature,
            coefficient,
        ])

coefficient_table = pd.DataFrame(
    coefficient_rows,
    columns=["Model", "Feature", "Standardized_Coefficient"],
)

coefficient_table


## 9. Classification threshold analysis

ROC-AUC evaluates ranking without committing to a classification cutoff. If the model is used to issue a binary warning, the threshold matters.

The following table illustrates the precision-recall tradeoff for Model 5 on the exploratory holdout. It is descriptive; a threshold should not be chosen solely because it performs best on this already-observed test sample.


In [ ]:
y_test = test["Recession_Next_12M"]
prob_5 = holdout_probabilities["Model 5"]

thresholds = np.arange(0.10, 0.81, 0.10)
threshold_rows = []

for threshold in thresholds:
    predictions = (prob_5 >= threshold).astype(int)

    threshold_rows.append([
        threshold,
        precision_score(y_test, predictions, zero_division=0),
        recall_score(y_test, predictions, zero_division=0),
        f1_score(y_test, predictions, zero_division=0),
        int(predictions.sum()),
    ])

threshold_table = pd.DataFrame(
    threshold_rows,
    columns=[
        "Threshold",
        "Precision",
        "Recall",
        "F1",
        "Positive_Predictions",
    ],
)

threshold_table


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(threshold_table["Threshold"], threshold_table["Precision"], marker="o", label="Precision")
ax.plot(threshold_table["Threshold"], threshold_table["Recall"], marker="o", label="Recall")
ax.plot(threshold_table["Threshold"], threshold_table["F1"], marker="o", label="F1")

ax.set_title("Model 5: Classification Threshold Tradeoff")
ax.set_xlabel("Probability threshold")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.legend()
plt.show()


## 10. Purged expanding-window walk-forward validation

This is the central validation exercise.

A normal random train/test split is inappropriate for a forecasting problem because it allows future observations to help predict the past.

We instead use expanding training windows and five-year test windows.

### Why purge 12 months?

The label at month \(t\) depends on outcomes through \(t+12\). If training ended immediately before a test period, the final training labels would contain information from inside the test period.

Therefore, the **12 months immediately preceding each test window are excluded from training**.

All five models are evaluated on the same dates, making the comparison apples-to-apples.


In [ ]:
periods = [
    ("2000-01-01", "2004-12-01"),
    ("2005-01-01", "2009-12-01"),
    ("2010-01-01", "2014-12-01"),
    ("2015-01-01", "2019-12-01"),
    ("2020-01-01", "2024-12-01"),
]

walk_forward_rows = []
oos_predictions = []

for start, end in periods:
    test_start = pd.Timestamp(start)

    # Purge the 12 months before each test period.
    train_end = test_start - pd.DateOffset(months=12)

    fold_train = model_data.loc[
        model_data.index < train_end
    ].copy()

    fold_test = model_data.loc[start:end].copy()

    y_train = fold_train["Recession_Next_12M"]
    y_test = fold_test["Recession_Next_12M"]

    for model_name, features in MODEL_SPECS.items():
        pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("logit", LogisticRegression(max_iter=1000)),
        ])

        pipeline.fit(fold_train[features], y_train)

        probabilities = pipeline.predict_proba(
            fold_test[features]
        )[:, 1]

        # AUC is undefined if a test window contains only one class.
        if y_test.nunique() == 2:
            auc = roc_auc_score(y_test, probabilities)
        else:
            auc = np.nan

        walk_forward_rows.append([
            model_name,
            start,
            end,
            len(fold_train),
            len(fold_test),
            int(y_test.sum()),
            auc,
        ])

        for date, actual, probability in zip(
            fold_test.index,
            y_test,
            probabilities,
        ):
            oos_predictions.append([
                date,
                model_name,
                int(actual),
                probability,
            ])

walk_forward_results = pd.DataFrame(
    walk_forward_rows,
    columns=[
        "Model",
        "Start",
        "End",
        "Training_Observations",
        "Test_Observations",
        "Positive_Target_Months",
        "ROC_AUC",
    ],
)

walk_forward_results


## 11. Walk-forward AUC comparison


In [ ]:
auc_pivot = walk_forward_results.pivot(
    index=["Start", "End"],
    columns="Model",
    values="ROC_AUC",
)

auc_pivot


In [ ]:
mean_fold_auc = (
    walk_forward_results
    .groupby("Model", as_index=False)["ROC_AUC"]
    .mean()
    .rename(columns={"ROC_AUC": "Mean_Fold_ROC_AUC"})
)

mean_fold_auc


The simple mean above treats each valid five-year fold equally. It is useful descriptively, but the folds differ in class composition and one fold may have no positive observations.

A second summary pools every genuinely out-of-sample monthly prediction from the walk-forward exercise and calculates one ROC-AUC for each model.


In [ ]:
oos_predictions = pd.DataFrame(
    oos_predictions,
    columns=[
        "Date",
        "Model",
        "Actual",
        "Probability",
    ],
)

pooled_auc_rows = []

for model_name in MODEL_SPECS:
    subset = oos_predictions[
        oos_predictions["Model"] == model_name
    ].copy()

    pooled_auc = roc_auc_score(
        subset["Actual"],
        subset["Probability"],
    )

    pooled_auc_rows.append([
        model_name,
        len(subset),
        int(subset["Actual"].sum()),
        pooled_auc,
    ])

pooled_auc_table = pd.DataFrame(
    pooled_auc_rows,
    columns=[
        "Model",
        "OOS_Observations",
        "Positive_Target_Months",
        "Pooled_Walk_Forward_ROC_AUC",
    ],
)

pooled_auc_table


## 12. Visualize walk-forward performance by period


In [ ]:
plot_data = walk_forward_results.dropna(subset=["ROC_AUC"]).copy()

fig, ax = plt.subplots(figsize=(12, 6))

for model_name in MODEL_SPECS:
    subset = plot_data[plot_data["Model"] == model_name]

    ax.plot(
        subset["Start"],
        subset["ROC_AUC"],
        marker="o",
        label=model_name,
    )

ax.axhline(0.5, linestyle="--", linewidth=1, label="Random-ranking benchmark")
ax.set_title("Purged Walk-Forward ROC-AUC by Test Period")
ax.set_xlabel("Test period start")
ax.set_ylabel("ROC-AUC")
ax.set_ylim(0, 1)
ax.legend(ncol=2)
plt.show()


## 13. Investigate the 2020–2024 regime

The pandemic period is economically unusual. A useful forecasting project should examine model failures rather than only reporting its strongest period.

The next table compares Model 5's out-of-sample probabilities with the realized target during 2020–2024.


In [ ]:
pandemic_period = oos_predictions[
    (oos_predictions["Model"] == "Model 5")
    & (oos_predictions["Date"] >= "2020-01-01")
    & (oos_predictions["Date"] <= "2024-12-01")
].copy()

pandemic_period.head(20)


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(
    pandemic_period["Date"],
    pandemic_period["Probability"],
    label="Model 5 probability",
)

ax.fill_between(
    pandemic_period["Date"],
    0,
    1,
    where=(pandemic_period["Actual"] == 1),
    alpha=0.15,
    label="Recession within next 12 months",
)

ax.set_title("Model 5 Out-of-Sample Predictions: 2020–2024")
ax.set_xlabel("Year")
ax.set_ylabel("Predicted probability")
ax.set_ylim(0, 1)
ax.legend()
plt.show()


## 14. Final model fit and current-data output

After validation is complete, Model 5 can be refit on all labeled historical observations. This does **not** retroactively change the validation results.

The latest available feature vector is then scored as an illustrative current model output. Because macroeconomic series are released on different schedules and may be revised, the date and underlying values should always be shown alongside the probability.


In [ ]:
final_features = MODEL_SPECS["Model 5"]

final_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logit", LogisticRegression(max_iter=1000)),
])

final_model.fit(
    model_data[final_features],
    model_data["Recession_Next_12M"],
)

# Find the latest month for which every Model 5 feature is available.
latest_feature_data = data.dropna(subset=final_features).copy()
latest_date = latest_feature_data.index.max()
latest_X = latest_feature_data.loc[[latest_date], final_features]

latest_probability = final_model.predict_proba(latest_X)[:, 1][0]

print("Latest feature date:", latest_date.date())
print()
print("Latest feature values:")
display(latest_X)
print()
print(f"Model 5 estimated probability: {latest_probability:.1%}")


## 15. Conclusions

This project examined whether combining multiple macroeconomic indicators improves 12-month U.S. recession forecasting relative to a yield-curve-only baseline.

### Main findings

The results show that additional macroeconomic information can substantially improve out-of-sample recession-risk ranking, although adding more predictors does not automatically improve performance.

The yield-curve-only model produced a pooled walk-forward ROC-AUC of **0.550**. Adding the 3-month change in unemployment increased ROC-AUC to **0.823**, representing the largest improvement between consecutive specifications.

Adding housing-start growth did not improve pooled performance, with Model 3 producing an ROC-AUC of **0.821**. Adding consumer sentiment subsequently increased ROC-AUC to **0.836**, while the full five-variable model incorporating the corporate credit spread achieved the highest pooled walk-forward ROC-AUC of **0.853**.

These results suggest that combining information from the yield curve, labor market, housing market, household expectations, and credit conditions can provide substantially more useful recession-risk signals than relying on the yield curve alone.

However, performance was not stable across historical periods. In particular, the models performed substantially worse during 2020–2024. This illustrates an important challenge in macroeconomic forecasting: relationships learned from previous business cycles may become less reliable during unusual economic regimes.

### Methodological limitations

- U.S. recessions are rare, leaving relatively few independent downturn episodes for model evaluation.

- The 12-month forecasting target creates overlapping labels across neighboring months.

- Macroeconomic data can be revised. This notebook uses currently available historical FRED observations rather than the exact data vintages that would have been available to forecasters in real time.

- A stricter real-time forecasting system could use **ALFRED historical vintages** and account explicitly for publication lags so that each historical prediction uses only information that was actually available on that date.

- ROC-AUC measures the model's ability to rank recession risk, not the accuracy or calibration of its predicted probabilities. Therefore, an ROC-AUC of 0.853 should not be interpreted as the model being 85.3% accurate.

- Logistic regression imposes a particular functional relationship between the predictors and recession risk.

- Estimated coefficients represent predictive associations and should not be interpreted as causal effects.

- Structural breaks and unusual economic shocks can reduce the stability of relationships estimated from historical data.

### Takeaway

The five-variable specification achieved the strongest pooled out-of-sample performance, with a ROC-AUC of **0.853**, compared with **0.550** for the yield-curve-only baseline. More importantly, the results show that both the information included in the model and the economic regime being forecast matter for predictive performance.

The project demonstrates an end-to-end empirical forecasting workflow:

**economic hypothesis → feature engineering → nested model comparison → time-aware validation → leakage control → out-of-sample evaluation → failure analysis**

Rather than assuming that the most complex model would perform best, the analysis tested each additional source of macroeconomic information and documented both improvements and periods in which the historical relationships broke down.